# TrainWatch: Getting Started

This notebook shows the simplest way to send training notifications with TrainWatch.
The PyPI distribution name is `trainwatch-notify`, but the import package is `trainwatch`.

In [ ]:
# If TrainWatch is not installed yet:
# !pip install trainwatch-notify

## Optional: Cloud email setup (no SMTP)
Set your Cloudflare Worker URL and register your email once.

In [ ]:
import os
from trainwatch import add_email

os.environ["TRAINWATCH_BASE_URL"] = "https://your-worker.workers.dev"

# Run once to register your email and save the API key locally.
# add_email("you@example.com")

## Basic training run (manual try/except)

In [ ]:
from trainwatch import monitor

def train_model(epochs=3):
    for epoch in range(1, epochs + 1):
        loss = 0.6 / epoch
        val_acc = 0.75 + (epoch * 0.03)
        monitor.log(epoch=epoch, loss=loss, val_acc=val_acc)

monitor.start()
try:
    train_model(epochs=5)
    monitor.end()
except Exception as exc:
    monitor.fail(exc)
    raise

## Context manager (auto end/fail)

In [ ]:
from trainwatch import monitor

def train_quick():
    for epoch in range(1, 4):
        monitor.log(epoch=epoch, loss=0.5 / epoch)

with monitor.watch():
    train_quick()

## Mid-run notifications

In [ ]:
import time
from trainwatch import monitor

monitor.start()
monitor.heartbeat(interval_seconds=60, message="Training still running")

for epoch in range(1, 4):
    time.sleep(1)
    monitor.step(notify_every=2, message=f"Epoch {epoch} completed")

monitor.end()

## Failure example (sends "Training Failed")

In [ ]:
from trainwatch import monitor

monitor.start()
try:
    # Uncomment to test failure notifications:
    # raise RuntimeError("Intentional failure example")
    monitor.end()
except Exception as exc:
    monitor.fail(exc)
    raise